In [1]:
import sys, os
from pathlib import Path
import torch
import torch.nn.functional as F
from torch.nn import CTCLoss, CrossEntropyLoss
from torch.utils.data import DataLoader
from itertools import chain
from tqdm import tqdm

PROJECT_PATH = Path(r"B:\College\DL\handwriting_autocomplete_system\phase3_style_transfer")
os.chdir(PROJECT_PATH)
sys.path.insert(0, str(PROJECT_PATH))

from lib.datasets import get_dataset, get_collect_fn
from lib.alphabet import strLabelConverter, get_lexicon, get_true_alphabet
from lib.utils import yaml2config
from networks.BigGAN_networks import Generator, Discriminator, PatchDiscriminator
from networks.module import Recognizer, WriterIdentifier, StyleEncoder, StyleBackbone
from networks.loss import recn_l1_loss, CXLoss, KLloss
from networks.rand_dist import prepare_z_dist, prepare_y_dist
from networks.utils import get_scheduler, idx_to_words, set_requires_grad, extract_all_patches

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
# Config
cfg = yaml2config(str(PROJECT_PATH / 'configs' / 'gan_iam.yml'))
cfg.device = str(device)
cfg.training.batch_size = 24
cfg.training.epochs = 70

# Dataset
collect_fn = get_collect_fn(cfg.training.sort_input, sort_style=True)
train_dataset = get_dataset(cfg.dataset, cfg.training.dset_split, recogn_aug=True, wid_aug=True, process_style=True)
train_loader = DataLoader(train_dataset, batch_size=cfg.training.batch_size, shuffle=True, collate_fn=collect_fn, num_workers=0, drop_last=True)
print(f"Training samples: {len(train_dataset)}, batches: {len(train_loader)}")

Training samples: 52231, batches: 2176


In [3]:
# Models
generator = Generator(**cfg.GenModel).to(device)
discriminator = Discriminator(**cfg.DiscModel).to(device)
patch_discriminator = PatchDiscriminator(**cfg.PatchDiscModel).to(device)
style_backbone = StyleBackbone(**cfg.StyBackbone).to(device)
style_encoder = StyleEncoder(**cfg.EncModel).to(device)
recognizer = Recognizer(**cfg.OcrModel).to(device)
writer_identifier = WriterIdentifier(**cfg.WidModel).to(device)

# Load pretrained OCR & WID
if os.path.exists(cfg.training.pretrained_r):
    recognizer.load_state_dict(torch.load(cfg.training.pretrained_r, map_location=device)['Recognizer'])
if os.path.exists(cfg.training.pretrained_w):
    w_dict = torch.load(cfg.training.pretrained_w, map_location=device)
    writer_identifier.load_state_dict(w_dict['WriterIdentifier'])
    style_backbone.load_state_dict(w_dict['StyleBackbone'])

# Freeze auxiliary networks
for p in recognizer.parameters(): p.requires_grad = False
for p in writer_identifier.parameters(): p.requires_grad = False
print("Models initialized")

Models initialized


In [4]:
# Optimizers
optimizer_G = torch.optim.Adam(chain(generator.parameters(), style_encoder.parameters()), lr=cfg.training.lr, betas=(cfg.training.adam_b1, cfg.training.adam_b2))
optimizer_D = torch.optim.Adam(chain(discriminator.parameters(), patch_discriminator.parameters()), lr=cfg.training.lr * 0.5, betas=(cfg.training.adam_b1, cfg.training.adam_b2))
lr_scheduler_G = get_scheduler(optimizer_G, cfg.training)
lr_scheduler_D = get_scheduler(optimizer_D, cfg.training)

# Loss functions
ctc_loss = CTCLoss(zero_infinity=True, reduction='mean')
classify_loss = CrossEntropyLoss()
contextual_loss = CXLoss()

# Lexicon
lexicon = get_lexicon(cfg.training.lexicon, get_true_alphabet(cfg.dataset), max_length=cfg.training.max_word_len)
if not lexicon:
    alphabet = get_true_alphabet(cfg.dataset)
    lexicon = sorted(set(''.join(chr(c) for c in train_dataset.lbs[s:s+l] if chr(c) in alphabet).lower() 
                         for s, l in zip(train_dataset.lb_seek_idxs, train_dataset.lb_lens) if 1 < l < cfg.training.max_word_len))

# Random distributions
z_dist = prepare_z_dist(cfg.training.batch_size, cfg.EncModel.style_dim, device, seed=cfg.seed)
y_dist = prepare_y_dist(cfg.training.batch_size, len(lexicon), device, seed=cfg.seed)
label_converter = strLabelConverter('all')
print(f"Lexicon: {len(lexicon)} words")

Lexicon: 465593 words


In [6]:
# Training loop
vae_mode = cfg.training.vae_mode
ctc_len_scale = recognizer.len_scale
history = {'epoch': [], 'g_loss': [], 'd_loss': []}
iter_count = 0

for epoch in range(1, cfg.training.epochs + 1):
    generator.train(); discriminator.train(); patch_discriminator.train(); style_encoder.train()
    epoch_g_loss, epoch_d_loss = 0.0, 0.0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{cfg.training.epochs}')
    for batch in pbar:
        real_imgs = batch['style_imgs'].to(device)
        real_img_lens = batch['style_img_lens'].to(device)
        real_aug_imgs = batch['aug_imgs'].to(device)
        real_aug_img_lens = batch['aug_img_lens'].to(device)
        real_lbs = batch['lbs'].to(device)
        real_lb_lens = batch['lb_lens'].to(device)
        real_wids = batch['wids'].to(device)
        max_label_len = real_lbs.size(-1)

        # === Train Discriminator ===
        optimizer_D.zero_grad()
        set_requires_grad([generator, style_encoder], False)
        set_requires_grad([discriminator, patch_discriminator], True)
        
        with torch.no_grad():
            y_dist.sample_()
            fake_words = idx_to_words(y_dist, lexicon, max_label_len, cfg.training.capitalize_ratio, cfg.training.blank_ratio)
            fake_lbs, fake_lb_lens = label_converter.encode(fake_words, max_label_len)
            fake_lbs, fake_lb_lens = fake_lbs.to(device), fake_lb_lens.to(device)
            z_dist.sample_()
            fake_imgs = generator(z_dist, fake_lbs, fake_lb_lens)
            enc_z = style_encoder(real_imgs, real_img_lens, style_backbone, vae_mode=vae_mode) if not vae_mode else style_encoder(real_imgs, real_img_lens, style_backbone, vae_mode=True)[0]
            style_imgs = generator(enc_z, fake_lbs, fake_lb_lens)
            recn_imgs = generator(enc_z, real_lbs, real_lb_lens)
            cat_fake = torch.cat([fake_imgs, style_imgs, recn_imgs], dim=0)
            cat_lb_lens = torch.cat([fake_lb_lens, fake_lb_lens, real_lb_lens], dim=0)

        fake_disc = discriminator(cat_fake.detach(), cat_lb_lens * cfg.char_width, cat_lb_lens)
        fake_patch = patch_discriminator(extract_all_patches(cat_fake, cat_lb_lens * cfg.char_width).detach())
        real_disc = discriminator(real_imgs, real_img_lens, real_lb_lens)
        real_disc_aug = discriminator(real_aug_imgs, real_aug_img_lens, real_lb_lens)
        real_patch = patch_discriminator(torch.cat([extract_all_patches(real_imgs, real_img_lens), extract_all_patches(real_aug_imgs, real_aug_img_lens)], dim=0))
        
        d_loss = (F.relu(1 + fake_disc).mean() + F.relu(1 + fake_patch).mean() + 
                  (F.relu(1 - real_disc).mean() + F.relu(1 - real_disc_aug).mean()) / 2 + F.relu(1 - real_patch).mean())
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(discriminator.parameters(), 5.0)
        torch.nn.utils.clip_grad_norm_(patch_discriminator.parameters(), 5.0)
        optimizer_D.step()
        epoch_d_loss += d_loss.item()

        # === Train Generator ===
        if iter_count % cfg.training.num_critic_train == 0:
            optimizer_G.zero_grad()
            set_requires_grad([discriminator, patch_discriminator], False)
            set_requires_grad([generator, style_encoder], True)
            
            y_dist.sample_()
            fake_words = idx_to_words(y_dist, lexicon, max_label_len, cfg.training.capitalize_ratio, cfg.training.blank_ratio, sort=True)
            fake_lbs, fake_lb_lens = label_converter.encode(fake_words, max_label_len)
            fake_lbs, fake_lb_lens = fake_lbs.to(device), fake_lb_lens.to(device)
            z_dist.sample_()
            fake_imgs = generator(z_dist, fake_lbs, fake_lb_lens)
            
            if vae_mode:
                (enc_z, mu, logvar), real_feats = style_encoder(real_imgs, real_img_lens, style_backbone, ret_feats=True, vae_mode=True)
            else:
                enc_z, real_feats = style_encoder(real_imgs, real_img_lens, style_backbone, ret_feats=True, vae_mode=False)
            
            style_imgs = generator(enc_z, fake_lbs, fake_lb_lens)
            recn_imgs = generator(enc_z, real_lbs, real_lb_lens)
            
            cat_fake = torch.cat([fake_imgs, style_imgs, recn_imgs], dim=0)
            cat_lb_lens = torch.cat([fake_lb_lens, fake_lb_lens, real_lb_lens], dim=0)
            adv_loss = -discriminator(cat_fake, cat_lb_lens * cfg.char_width, cat_lb_lens).mean()
            adv_patch = -patch_discriminator(extract_all_patches(cat_fake, cat_lb_lens * cfg.char_width)).mean()
            
            # CTC losses
            ctc_rand = ctc_loss(recognizer(fake_imgs, fake_lb_lens * cfg.char_width), fake_lbs, fake_lb_lens * cfg.char_width // ctc_len_scale, fake_lb_lens)
            ctc_style = ctc_loss(recognizer(style_imgs, fake_lb_lens * cfg.char_width), fake_lbs, fake_lb_lens * cfg.char_width // ctc_len_scale, fake_lb_lens)
            ctc_recn = ctc_loss(recognizer(recn_imgs, real_lb_lens * cfg.char_width), real_lbs, real_lb_lens * cfg.char_width // ctc_len_scale, real_lb_lens)
            
            info_loss = (style_encoder(fake_imgs, fake_lb_lens * cfg.char_width, style_backbone) - z_dist.detach()).abs().mean()
            recn_loss = recn_l1_loss(recn_imgs, real_imgs, real_img_lens)
            
            cat_style = torch.cat([style_imgs, recn_imgs], dim=0)
            wid_logits, fake_feats = writer_identifier(cat_style, torch.cat([fake_lb_lens, real_lb_lens]) * cfg.char_width, style_backbone, ret_feats=True)
            wid_loss = classify_loss(wid_logits, real_wids.repeat(2))
            
            ctx_loss = sum(contextual_loss(r, f) for r, f in zip(real_feats, [ff.chunk(2)[0] for ff in fake_feats]))
            kl_loss = KLloss(mu, logvar) if vae_mode else torch.tensor(0.0, device=device)
            
            g_loss = adv_loss + adv_patch + 3.0 * (ctc_rand + ctc_style + ctc_recn) + 1.5 * info_loss + 1.5 * wid_loss + 5.0 * recn_loss + cfg.training.lambda_ctx * ctx_loss + cfg.training.lambda_kl * kl_loss
            g_loss.backward()
            torch.nn.utils.clip_grad_norm_(generator.parameters(), 5.0)
            torch.nn.utils.clip_grad_norm_(style_encoder.parameters(), 5.0)
            optimizer_G.step()
            epoch_g_loss += g_loss.item()
        
        iter_count += 1
        pbar.set_postfix({'D': f'{d_loss.item():.3f}', 'G': f'{g_loss.item():.3f}' if iter_count % cfg.training.num_critic_train == 0 else '-'})
    
    # Epoch end
    n_g = max(1, len(train_loader) // cfg.training.num_critic_train)
    history['epoch'].append(epoch)
    history['g_loss'].append(epoch_g_loss / n_g)
    history['d_loss'].append(epoch_d_loss / len(train_loader))
    print(f"Epoch {epoch}: G={history['g_loss'][-1]:.4f}, D={history['d_loss'][-1]:.4f}")
    
    # Save checkpoint
    os.makedirs('checkpoints', exist_ok=True)
    torch.save({
        'epoch': epoch, 'iter_count': iter_count, 'history': history,
        'generator': generator.state_dict(), 'style_encoder': style_encoder.state_dict(),
        'discriminator': discriminator.state_dict(), 'patch_discriminator': patch_discriminator.state_dict(),
        'optimizer_G': optimizer_G.state_dict(), 'optimizer_D': optimizer_D.state_dict(),
    }, f'checkpoints/epoch_{epoch}.pth')
    
    lr_scheduler_G.step(epoch)
    lr_scheduler_D.step(epoch)

print("Training complete!")

Epoch 1/70:   0%|          | 0/2176 [00:00<?, ?it/s]



AcceleratorError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
